# Deep Retention: entrenamiento en Colab

Entrena la red, evalúa, calcula SHAP y descarga `artefactos.zip`.

**Entorno:** basta con CPU. Ejecute las celdas en orden. Antes, cree `colab_bundle.zip` con `python -m scripts.empaquetar_colab`.

## 0. Comprobar Python

In [ ]:
import sys

version = sys.version.split()[0]
print(f"Python {version}")
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise SystemExit(f"Este cuaderno admite Python 3.12 y 3.13, y este entorno usa {version}. No ejecute las demás celdas.")

## 1. Subir el paquete

In [ ]:
from pathlib import Path

from google.colab import files

subidos = files.upload()
if len(subidos) != 1:
    raise SystemExit("Suba un único archivo: colab_bundle.zip")
# Otra subida llega como «colab_bundle (1).zip»: se guarda lo recibido con el nombre fijo, sin mirar el nombre.
Path("colab_bundle.zip").write_bytes(next(iter(subidos.values())))

## 2. Descomprimir con seguridad

Solo se aceptan los archivos esperados del paquete.

In [ ]:
import re
import stat
import zipfile
from pathlib import Path

PROYECTO = Path("proyecto").resolve()
PERMITIDOS = re.compile(r"src/[A-Za-z0-9_]+\.py|data/Customer-Churn-Records\.csv|requirements\.txt")


def extraer_paquete(ruta_zip, destino):
    with zipfile.ZipFile(ruta_zip) as paquete:
        entradas = paquete.infolist()
        for entrada in entradas:
            if not PERMITIDOS.fullmatch(entrada.filename):
                raise ValueError(f"Entrada no permitida: {entrada.filename!r}")
            if stat.S_IFMT(entrada.external_attr >> 16) not in (0, stat.S_IFREG):
                raise ValueError(f"No es un archivo normal: {entrada.filename!r}")
        for entrada in entradas:
            salida = Path(destino) / entrada.filename
            salida.parent.mkdir(parents=True, exist_ok=True)
            salida.write_bytes(paquete.read(entrada))


extraer_paquete("colab_bundle.zip", PROYECTO)

## 3. Verificar los datos

In [ ]:
import hashlib

SHA256_DATOS = "2abdc051d9170540777bf35652ccb02a6ce9c9645892b078762cb729039e44d2"
obtenido = hashlib.sha256((PROYECTO / "data/Customer-Churn-Records.csv").read_bytes()).hexdigest()
if obtenido != SHA256_DATOS:
    raise SystemExit(f"SHA-256 incorrecto: {obtenido}")
print("Datos verificados")

## 4. Instalar versiones fijadas

Cada etapa corre en un subproceso nuevo, así que no hace falta reiniciar el entorno.

In [ ]:
import os
import subprocess
import sys

ENTORNO = {
    **os.environ,
    "KERAS_BACKEND": "tensorflow",
    "MPLBACKEND": "Agg",
    "PYTHONHASHSEED": "42",
    "PYTHONUNBUFFERED": "1",
    "PYTHONUTF8": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
}


def ejecutar(*modulo):
    """Corre `python -m ...`, muestra cada línea al llegar y falla si el proceso falla."""
    proceso = subprocess.Popen(
        [sys.executable, "-m", *modulo],
        cwd=PROYECTO,
        env=ENTORNO,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    try:
        for linea in proceso.stdout:
            print(linea, end="", flush=True)
        proceso.wait()
    except BaseException:  # interrumpir la celda no debe dejar la etapa corriendo
        proceso.kill()
        raise
    if proceso.returncode != 0:
        raise RuntimeError(f"Falló {modulo[0]} (código {proceso.returncode})")


ejecutar("pip", "install", "--require-hashes", "-r", "requirements.txt")

## 5. Entrenar, evaluar y explicar

SHAP es la etapa más lenta (varios minutos en CPU).

In [ ]:
for etapa in ["src.train", "src.evaluate", "src.explain"]:
    print(f"--- {etapa} ---", flush=True)
    ejecutar(etapa)

## 6. Descargar los artefactos

Después, en el proyecto: `python -m scripts.importar_artefactos artefactos.zip`.

In [ ]:
ARTEFACTOS = ["modelo.keras", "preprocesador.json", "calibrador.json", "metricas.json", "shap_global.json", "fondo_shap.json"]

with zipfile.ZipFile("artefactos.zip", "w", zipfile.ZIP_DEFLATED) as salida:
    for nombre in ARTEFACTOS:
        salida.write(PROYECTO / "models" / nombre, nombre)
    for figura in sorted((PROYECTO / "reports" / "figuras").glob("*.png")):
        salida.write(figura, f"figuras/{figura.name}")

files.download("artefactos.zip")